In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

In [ ]:
# Load images
img1 = cv2.imread('graf/graf/img1.ppm')
img5 = cv2.imread('graf/graf/img5.ppm')

# Convert to RGB
img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img5_rgb = cv2.cvtColor(img5, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('Image 1')
plt.imshow(img1_rgb)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Image 5')
plt.imshow(img5_rgb)
plt.axis('off')
plt.show()

(a) Compute and match SIFT features between the two images.

In [ ]:
# SIFT
sift = cv2.SIFT_create()

# Find keypoints and descriptors
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img5, None)

# FLANN parameters
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)

# Ratio test
good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

# Draw matches
match_img = cv2.drawMatches(img1, kp1, img5, kp2, good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

plt.figure(figsize=(15, 15))
plt.imshow(match_img)
plt.title('SIFT Feature Matches')
plt.axis('off')
plt.show()

(b) Compute the homography using your own code within RANSAC and compare with the homography given in the dataset.

In [ ]:
def compute_homography(src_pts, dst_pts):
    A = []
    for i in range(len(src_pts)):
        x, y = src_pts[i][0], src_pts[i][1]
        u, v = dst_pts[i][0], dst_pts[i][1]
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    A = np.asarray(A)
    U, S, Vh = np.linalg.svd(A)
    L = Vh[-1, :] / Vh[-1, -1]
    H = L.reshape(3, 3)
    return H

def ransac(matches, kp1, kp2, threshold, iterations):
    best_H = None
    max_inliers = 0

    for _ in range(iterations):
        # Randomly select 4 matches
        sample_matches = random.sample(matches, 4)
        
        src_pts = np.float32([kp1[m.queryIdx].pt for m in sample_matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in sample_matches]).reshape(-1, 1, 2)

        # Compute homography
        H = compute_homography(src_pts.reshape(-1, 2), dst_pts.reshape(-1, 2))

        inliers = 0
        for m in matches:
            pt1 = np.array([kp1[m.queryIdx].pt[0], kp1[m.queryIdx].pt[1], 1])
            pt2_actual = np.array([kp2[m.trainIdx].pt[0], kp2[m.trainIdx].pt[1]])
            
            pt2_transformed = np.dot(H, pt1)
            pt2_transformed /= pt2_transformed[2]
            
            dist = np.linalg.norm(pt2_actual - pt2_transformed[:2])
            if dist < threshold:
                inliers += 1
        
        if inliers > max_inliers:
            max_inliers = inliers
            best_H = H
            
    return best_H

# RANSAC parameters
threshold = 5.0
iterations = 1000

src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

# Compute homography with RANSAC
H_ransac = ransac(good_matches, kp1, kp2, threshold, iterations)

# Homography from dataset
H_given = np.loadtxt('graf/graf/H1to5p')

print("Homography from RANSAC:")
print(H_ransac)
print("\nHomography from dataset:")
print(H_given)

(c) Stitch img1.ppm onto img5.ppm.

In [ ]:
# Warp img1 to img5's perspective
h, w, _ = img5.shape
warped_img1 = cv2.warpPerspective(img1, H_ransac, (w, h))

# Create a mask of the warped image
mask = np.zeros_like(img5, dtype='uint8')
mask[warped_img1 > 0] = 255

# Invert the mask
mask_inv = cv2.bitwise_not(mask)

# Black out the area of the warped image in the target image
img5_bg = cv2.bitwise_and(img5, mask_inv)

# Combine the two images
result = cv2.add(img5_bg, warped_img1)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title('Stitched Image')
plt.axis('off')
plt.show()